In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

import copy
from torch.utils.data import DataLoader, TensorDataset

    
        
import matplotlib.pyplot as plt
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

import copy
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
from data_loading import *
from data_generation import *
from data_plot import *
from models import *

In [ ]:
class UnifiedConditionalEstimatorCVAR(nn.Module):
    """
    Classe maîtresse unifiant K Flux (Flows) avec agrégation Softmin pour 
    modéliser une union de régions de confiance avec matrices de précision libres.
    """
    def __init__(self, dim_X, dim_y, K=3, det_normalized=True, use_partition=True, hidden_dim=64, n_hidden_layer=2, n_hidden_layer_quantile=1, num_flow_layers=1, cov_mode="full_cholesky", dropout=0.1, dropout_quantile=0.3):
        super().__init__()
        self.d = dim_y
        self.K = K       
        self.det_normalized = det_normalized
        self.external_quantile = False # Whether to use our quantile function or another model to predict the quantiles
        
        ### Create the share network for center, matrix and flow
        layers = []
        layers.extend([
            nn.Linear(dim_X, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout)
        ])
        for _ in range(n_hidden_layer - 1):
            layers.extend([
                nn.Linear(hidden_dim, hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.ReLU(),
                nn.Dropout(p=dropout)
            ])
        layers.append(nn.Linear(hidden_dim, hidden_dim))
        self.shared_net = nn.Sequential(*layers)

        ### All the centers 
        self.head_mu = nn.Linear(hidden_dim, self.K * self.d)

        ### All the matrices
        if det_normalized:
            self.head_precisions = nn.ModuleList([
                RobustPrecisionHead(hidden_dim, self.d, mode=cov_mode)
                for _ in range(K)
            ])
        else:
            self.head_precisions = nn.ModuleList([
                RobustPrecisionHeadWithDet(hidden_dim, self.d, mode=cov_mode)
                for _ in range(K)
            ])
        
        ### Declare the volume preserving flows
        self.flows = nn.ModuleList([
            ConditionalVolumePreservingFlow(self.d, dim_X, num_layers=num_flow_layers, dropout=dropout) 
            for _ in range(K)
        ])

        ### Use the partition of the space only when asked and using multiple flows without determinant
        if K == 1 or not det_normalized: 
            use_partition = False
        self.use_partition = use_partition
        if use_partition:
            self.partition_score = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim // 2),
                nn.ReLU(),
                nn.Linear(hidden_dim // 2, K),
                nn.Softmax(dim=-1)
            )
        else:
            self.partition_score = None
        
        ### Quantile manager to predict the quantiles
        self.quantile_manager = MultipleQuantiles(dim_X=dim_X, 
                                             hidden_dim=hidden_dim, 
                                             n_hidden_layer_quantile=n_hidden_layer_quantile, 
                                             dropout_quantile=dropout_quantile
                                             )
        self.q_alpha = None

    def forward(self, X):
        h = self.shared_net(X)
        B = X.shape[0]
        
        mu_flat = self.head_mu(h)
        mu = mu_flat.view(B, self.K, self.d)
    
        if self.external_quantile:
            q = self.external_quantile_model.predict(X, output_type="quantiles", alphas=[self.tau])
            q = torch.tensor(q, dtype=torch.float32)
            q = q.view(-1, 1)
        else:
            q = self.quantile_manager.forward_tau(X)

    
        precision_params_list = [head(h) for head in self.head_precisions]

        if self.K > 1 and self.det_normalized and self.use_partition:
            partition = self.partition_score(h)
            # raw_partition = self.partition_score(h)
            # eps = 1e-3
            # partition = raw_partition * (1 - eps) + (eps / self.K) 
        else:
            partition = None
        
        return mu, precision_params_list, partition, q
    
    def get_q_kernel(self, X):
        raw_q_low, raw_q_high = self.quantile_manager.forward_bound(X)
    
        q_low = torch.min(raw_q_low, raw_q_high)
        q_high = torch.max(raw_q_low, raw_q_high)

        return q_low, q_high
    
    def step_parameter(self):
        self.num_iteration += 1
        self.beta += self.lr_beta
        self.beta = min(self.beta_max, self.beta)
    
    def call_conformalize(self, X):
        if self.q_alpha is None:
            raise Exception("Call the conformalize first")
        mu, precision_params_list, partition, q = self(X)
        if self.conformal_score == "additive":
            return mu, precision_params_list, partition, q + self.q_alpha
        elif self.conformal_score == "multiplicative":
            return mu, precision_params_list, partition, q * self.q_alpha
            
    def get_cover(self, X, y):
        scores = self.get_normalized_scores(X, y)
        cover = scores <= self.q_alpha
        return cover*1.0
       
    def get_frontiers(self, X, y, use_conformalize=False):        
        if use_conformalize: mu, precision_params_list, partition, q = self.call_conformalize(X)
        else: mu, precision_params_list, partition, q = self(X)

        G_list = []
        log_det_L_list = []

        for k in range(self.K):
            z_k = self.flows[k](y, X)
            mu_k = mu[:, k, :]
            
            diff_k = (z_k - mu_k).unsqueeze(-1)
            
            if partition is not None:
                p_k = partition[:, k:k+1] + 1e-8 
            
            mode_k = precision_params_list[k][0]

            if mode_k == 'full_cholesky':
                L_k = precision_params_list[k][1]
                L_diff_k = torch.bmm(L_k, diff_k).squeeze(-1)
                G_k_raw = torch.sum(L_diff_k ** 2, dim=1)
                                
                idx = torch.arange(self.d, device=X.device)
                log_det_L_k_raw = torch.sum(torch.log(L_k[:, idx, idx]), dim=1)
                
            elif mode_k == 'low_rank':
                D_k, V_k = precision_params_list[k][1], precision_params_list[k][2]
                
                diff_sq_k = diff_k.squeeze(-1) ** 2
                G_k_raw = torch.sum(D_k * diff_sq_k, dim=1) + torch.sum(torch.bmm(V_k.transpose(1, 2), diff_k).squeeze(-1) ** 2, dim=1)
                
                log_det_D_k = torch.sum(torch.log(D_k + 1e-6), dim=1)
                D_inv_V_k = V_k / (D_k.unsqueeze(-1) + 1e-6) 
                V_T_D_inv_V_k = torch.bmm(V_k.transpose(1, 2), D_inv_V_k) 
                I = torch.eye(V_k.shape[-1], device=X.device).unsqueeze(0).expand(X.shape[0], -1, -1)
                _, logdet_k = torch.linalg.slogdet(I + V_T_D_inv_V_k) 
                log_det_L_k_raw = 0.5 * (log_det_D_k + logdet_k)

            if partition is not None and self.tau_parameterAnnealer.warm_start_step < self.num_iteration:
                d = y.shape[-1]
                p_k_squeeze = p_k.squeeze(-1)
                # G_k = G_k_raw - torch.log(p_k_squeeze)
                # log_det_L_k = log_det_L_k_raw - torch.log(p_k_squeeze)
                G_k = G_k_raw / (p_k_squeeze )**(2.0 / d)
                regularization = max((self.tau_parameterAnnealer.warm_start_step - self.num_iteration), 0) * torch.log(p_k_squeeze) 
                # log_det_L_k = log_det_L_k_raw - torch.log(p_k_squeeze) + regularization - regularization.detach()
                log_det_L_k = log_det_L_k_raw - torch.log(p_k_squeeze) 
            else:
                G_k = G_k_raw 
                log_det_L_k = log_det_L_k_raw 

            G_list.append(G_k)
            log_det_L_list.append(log_det_L_k)

        log_det_L_stacked = torch.stack(log_det_L_list, dim=1)
        G_stacked = torch.stack(G_list, dim=1)

        if self.K > 1:
            # G = - (1.0 / self.beta) * torch.logsumexp(-self.beta * G_stacked, dim=1, keepdim=True)
            # Calculate softmax weights (dim=1 is the component dimension K)
            weights = torch.softmax(-self.beta * G_stacked, dim=1)
            G = torch.sum(weights * G_stacked, dim=1, keepdim=True)
        else:
            G = G_stacked # TODO: check this

        return G, log_det_L_stacked, q
    
    def get_det_L(self, X, use_conformalize=False):        
        if use_conformalize: mu, precision_params_list, partition, q = self.call_conformalize(X)
        else: mu, precision_params_list, partition, q = self(X)

        log_det_L_list = []

        for k in range(self.K):
            
            if partition is not None:
                p_k = partition[:, k:k+1] + 1e-8 
            
            mode_k = precision_params_list[k][0]

            if mode_k == 'full_cholesky':
                L_k = precision_params_list[k][1]
                idx = torch.arange(self.d, device=X.device)
                log_det_L_k_raw = torch.sum(torch.log(L_k[:, idx, idx]), dim=1)
                
            elif mode_k == 'low_rank':
                D_k, V_k = precision_params_list[k][1], precision_params_list[k][2]
                                
                log_det_D_k = torch.sum(torch.log(D_k + 1e-6), dim=1)
                D_inv_V_k = V_k / (D_k.unsqueeze(-1) + 1e-6) 
                V_T_D_inv_V_k = torch.bmm(V_k.transpose(1, 2), D_inv_V_k) 
                I = torch.eye(V_k.shape[-1], device=X.device).unsqueeze(0).expand(X.shape[0], -1, -1)
                _, logdet_k = torch.linalg.slogdet(I + V_T_D_inv_V_k) 
                log_det_L_k_raw = 0.5 * (log_det_D_k + logdet_k)

            if partition is not None and self.tau_parameterAnnealer.warm_start_step < self.num_iteration:
                p_k_squeeze = p_k.squeeze(-1)
                log_det_L_k = log_det_L_k_raw - torch.log(p_k_squeeze)
            else:
                d = mu.shape[-1]
                log_det_L_k = log_det_L_k_raw 
            log_det_L_list.append(log_det_L_k)

        log_det_L_stacked = torch.stack(log_det_L_list, dim=1)

        return log_det_L_stacked, q

    def get_normalized_scores(self, X, y, batch_size=1000):
        frontiers_list = []
        q_list = []

        with torch.no_grad():
            for i in range(0, len(X), batch_size):
                X_batch = X[i:i + batch_size]
                y_batch = y[i:i + batch_size]
                frontiers_batch, _, q_batch = self.get_frontiers(X_batch, y_batch)
                frontiers_list.append(frontiers_batch.detach()) 
                q_list.append(q_batch.detach()) 
                
            frontiers = torch.cat(frontiers_list, dim=0)
            q = torch.cat(q_list, dim=0)
        
        
        # S, _, q = self.get_frontiers(X, y)
        return (frontiers / q).squeeze() 

    def compute_volume(self, X, batch_size = 1_000):
        """
        Calcule le volume exact approché (somme des composantes) pour un (ou plusieurs) point(s) X.
        """
        #TODO
        if self.q_alpha is None:
            raise Exception("Call the conformalize first")
        self.eval()

        det_list = []
        q_list = []

        with torch.no_grad():
            for i in range(0, len(X), batch_size):
                X_batch = X[i:i + batch_size]
                det_L_batch, q_batch = self.get_det_L(X_batch, use_conformalize=True)
                det_list.append(det_L_batch.detach()) 
                q_list.append(q_batch.detach()) 
                
            log_det_L_stacked = torch.cat(det_list, dim=0)
            q_conformalize = torch.cat(q_list, dim=0)

            # log_det_L_stacked, q_conformalize = self.get_det_L(X, use_conformalize=True)

            log_vol_penalty = torch.logsumexp(-log_det_L_stacked, dim=1, keepdim=True)
            log_V_constant = math.log((math.pi ** (self.d / 2)) / math.gamma(self.d / 2 + 1))
            log_vol_k = log_V_constant + (self.d / 2) * torch.log(q_conformalize) + log_vol_penalty
            total_vol = torch.exp(log_vol_k)
            
        return total_vol.squeeze()
    
    def compute_average_volume(self, X, scaled=False):
        volumes = self.compute_volume(X)
        return torch.mean(volumes) if not scaled else torch.mean(volumes**(1/self.d))

    def conformalize(self, X_cal, y_cal, alpha, conformal_score="multiplicative", fake_for_trial=False):
        self.conformal_score = conformal_score
        if fake_for_trial:
            print('NO CONFORMALIZATION AS ASKED')
            if conformal_score == "multiplicative":
                self.q_alpha = 1.0
            elif conformal_score == "additive":
                self.q_alpha = 0.0
            else:
                raise ValueError("The conformal score is not well defined.")
            return
        self.eval()
        with torch.no_grad():
            n = X_cal.shape[0]
            
            scores = self.get_normalized_scores(X_cal, y_cal) 
            p = int(np.ceil((n + 1) * (1 - alpha)))
            scores = torch.sort(scores, descending=True).values

            self.q_alpha = scores[p].item()  
                                        
            print(f"Conformalisation terminée (n={n}, alpha={alpha}). Multiplicateur q_alpha = {self.q_alpha:.4f}")

    
    def fit(
            self, 
            X_train, 
            y_train, 
            tau, 
            X_val=None, 
            y_val=None, 
            epochs=1000, 
            lr=1e-3, 
            batch_size=32, 
            weight_decay=1e-4, 
            beta = 5,
            beta_max = 100,
            lr_beta=1e-3,
            loss_function="log_volume",
            return_best=True, 
            print_every=1, 
            tau_parameterAnnealer=None
            ):        

        self.loss_function = loss_function
        self.tau = tau

        self.num_iteration = 0
        self.beta = beta
        self.beta_max = beta_max
        self.lr_beta = lr_beta

        if tau_parameterAnnealer is None:
            print("Using default fit param")
            self.tau_parameterAnnealer = TauParameterAnnealer(tau)  
        else:
            self.tau_parameterAnnealer = tau_parameterAnnealer
        
        # Dataloaders preparation
        train_dataset = TensorDataset(X_train, y_train)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

        optimizer = torch.optim.AdamW(self.parameters(), lr=lr, weight_decay=weight_decay)
        
        has_val = X_val is not None and y_val is not None
        if has_val:
            val_dataset = TensorDataset(X_val, y_val)
            val_loader = DataLoader(val_dataset, batch_size=batch_size * 2, shuffle=False)

        # Keep track of the best model
        self.best_valid_vol = float('inf')
        self.best_model_weights = None

        for epoch in range(epochs):
            # Training phase
            self.train()
            total_train_loss = 0.0
            epoch_coverage = 0.0
            epoch_valid_volume = 0.0
            total_train_pb_loss = 0.0
            
            for batch_X, batch_y in train_loader:
            
                ### Treat the quantiles
                optimizer.zero_grad()
            
                loss_tau = self.compute_pb_loss(batch_X, batch_y, tau)
                loss_low, loss_high = self.compute_pb_kernel_loss(batch_X, batch_y)
                loss_pb = loss_low + loss_high + loss_tau
                loss_pb.backward()
                
                torch.nn.utils.clip_grad_norm_(self.parameters(), 1.0)
                optimizer.step()

                ### Treat the shape
                optimizer.zero_grad()
                
                loss, batch_coverage, batch_valid_volume = self.compute_loss(batch_X, batch_y, tau)
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.parameters(), 1.0)
                optimizer.step()

                ### Updated the parameters (beta, phi(n), psi(n))
                self.step_parameter()
                                
                total_train_loss += loss.item() * batch_X.size(0)
                epoch_coverage += batch_coverage.item() * batch_X.size(0)
                epoch_valid_volume += batch_valid_volume.item() * batch_X.size(0)
                total_train_pb_loss += loss_pb.item() * batch_X.size(0)
                
            avg_train_loss = total_train_loss / len(X_train)
            avg_train_cov = epoch_coverage / len(X_train)
            avg_train_vol = epoch_valid_volume / len(X_train)
            avg_train_pb_loss = total_train_pb_loss / len(X_train)

            # Early stopping is performed on the volume - used if no validation set
            current_target_vol = avg_train_vol 

            # Validation if possible
            if has_val:
                self.eval()
                val_epoch_coverage = 0.0
                val_epoch_valid_volume = 0.0
                total_val_loss = 0.0
                total_val_pb_loss = 0.0
                
                with torch.no_grad():
                    for v_batch_X, v_batch_y in val_loader:
                        
                        loss_pb = self.compute_pb_loss(v_batch_X, v_batch_y, tau)
                        loss, v_batch_coverage, v_batch_valid_volume = self.compute_loss(v_batch_X, v_batch_y, tau)

                        val_epoch_coverage += v_batch_coverage.item() * v_batch_X.size(0)
                        val_epoch_valid_volume += v_batch_valid_volume.item() * v_batch_X.size(0)
                        total_val_loss += loss.item() * v_batch_X.size(0)
                        total_val_pb_loss += loss_pb.item() * v_batch_X.size(0)
                        
                avg_val_cov = val_epoch_coverage / len(X_val)
                avg_val_vol = val_epoch_valid_volume / len(X_val)
                avg_val_loss = total_val_loss / len(X_val)
                avg_val_pb_loss = total_val_pb_loss / len(X_val)
                current_target_vol = avg_val_vol

            # Saving the dic of best models
            if current_target_vol < self.best_valid_vol:
                self.best_valid_vol = current_target_vol
                self.best_model_weights = copy.deepcopy(self.state_dict())

            # Logs
            if (epoch + 1) % print_every == 0 or epoch == epochs - 1:
                tau_low, tau_high = self.tau_parameterAnnealer.get_params()
                if has_val:
                    print(f"Epoch {epoch+1}/{epochs} | T-Loss: {avg_train_loss:.4f} | V-Loss: {avg_val_loss:.4f} | "
                          f"T-Cov: {avg_train_cov:.3f} | V-Cov: {avg_val_cov:.3f} | "
                          f"T-Vol: {avg_train_vol:.4f} | V-Vol: {avg_val_vol:.2f} | "
                          f"BEST-Vol: {self.best_valid_vol:.4f} | tau_low: {tau_low:.2f} | tau_high: {tau_high:.2f} | "
                          f"T-pb-loss: {avg_train_pb_loss:.2f} | V-pb-loss: {avg_val_pb_loss:.2f} | "
                          )

                else:
                    print(f"Epoch {epoch+1}/{epochs} | T-Loss: {avg_train_loss:.4f} | "
                          f"T-Cov: {avg_train_cov:.3f} | "
                          f"T-Vol: {avg_train_vol:.4f} | "
                          f"BEST-Vol: {self.best_valid_vol:.4f} | tau_low: {tau_low:.2f} | tau_high: {tau_high:.2f} | "
                          f"T-pb-loss: {avg_train_pb_loss:.2f} "
                          )

        # Loading best weights in the end if requested
        if self.best_model_weights is not None and return_best:
            print(f"Entraînement terminé. Restauration des meilleurs poids avec un Volume Valide de : {self.best_valid_vol:.4f}")
            self.load_state_dict(self.best_model_weights)
            
        return self
    
    def fit_external_quantile(
            self, 
            quantile_model,
            X_train, 
            y_train, 
            batch_size = 300,
            **kwargs
            ): 
        
        frontiers_list = []

        with torch.no_grad():
            for i in range(0, len(X_train), batch_size):
                X_batch = X_train[i:i + batch_size]
                y_batch = y_train[i:i + batch_size]
                frontiers_batch, _, _ = self.get_frontiers(X_batch, y_batch)
                frontiers_list.append(frontiers_batch.detach()) 
                # frontiers_list.append(frontiers_batch.detach().cpu()) 
            frontiers_train = torch.cat(frontiers_list, dim=0)
        
        self.external_quantile_model = quantile_model
        self.external_quantile_model.fit(X_train, frontiers_train.squeeze())
    
        self.external_quantile = True

    def fit_quantile(
            self, 
            X_train, 
            y_train, 
            tau, 
            X_val=None, 
            y_val=None, 
            epochs=1000, 
            lr=1e-3, 
            batch_size=32, 
            weight_decay=1e-4, 
            return_best=True, 
            print_every=1,
            restart=True
            ): 
        
        if restart:
            self.quantile_manager.reset_parameters()
        
        # Dataloaders preparation
        train_dataset = TensorDataset(X_train, y_train)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

        optimizer = torch.optim.AdamW(self.parameters(), lr=lr, weight_decay=weight_decay)
        
        has_val = X_val is not None and y_val is not None
        if has_val:
            val_dataset = TensorDataset(X_val, y_val)
            val_loader = DataLoader(val_dataset, batch_size=batch_size * 2, shuffle=False)

        # Keep track of the best model
        best_target_loss = float('inf')
        self.best_quantile_model_weights = None

        for epoch in range(epochs):
            # Training phase
            self.train()
            epoch_coverage = 0.0
            total_train_pb_loss = 0.0
            
            for batch_X, batch_y in train_loader:
            
                ### Treat the quantiles
                optimizer.zero_grad()
            
                loss_tau, coverage = self.compute_pb_loss(batch_X, batch_y, tau, with_coverage=True)
                loss_pb =  loss_tau
                loss_pb.backward()
                
                epoch_coverage += coverage
                torch.nn.utils.clip_grad_norm_(self.parameters(), 1.0)
                optimizer.step()

                total_train_pb_loss += loss_pb.item() * batch_X.size(0)
                
            avg_train_cov = epoch_coverage / len(X_train)
            avg_train_pb_loss = total_train_pb_loss / len(X_train)

            current_target_loss = avg_train_pb_loss

            # Validation if possible
            if has_val:
                self.eval()
                val_epoch_coverage = 0.0
                total_val_pb_loss = 0.0
                
                with torch.no_grad():
                    for v_batch_X, v_batch_y in val_loader:
                        
                        loss_pb, coverage = self.compute_pb_loss(v_batch_X, v_batch_y, tau, with_coverage=True)
                        total_val_pb_loss += loss_pb.item() * v_batch_X.size(0)
                        val_epoch_coverage += coverage
                        
                avg_val_cov = val_epoch_coverage / len(X_val)
                avg_val_pb_loss = total_val_pb_loss / len(X_val)

                current_target_loss = avg_val_pb_loss

            # Saving the dic of best models
            if current_target_loss < best_target_loss:
                best_target_loss = current_target_loss
                self.best_quantile_model_weights = copy.deepcopy(self.state_dict())

            # Logs
            if (epoch + 1) % print_every == 0 or epoch == epochs - 1:
                if has_val:
                    print(f"Epoch {epoch+1}/{epochs} | T-Loss: {avg_train_pb_loss:.4f} | V-Loss: {avg_val_pb_loss:.4f} | "
                          f"T-Cov: {avg_train_cov:.3f} | V-Cov: {avg_val_cov:.3f} | "
                          )
                else:
                    print(f"Epoch {epoch+1}/{epochs} | T-Loss: {avg_train_pb_loss:.4f} | "
                          f"T-Cov: {avg_train_cov:.3f} |"
                          )

        # Loading best weights in the end if requested
        if self.best_quantile_model_weights is not None and return_best:
            self.load_state_dict(self.best_quantile_model_weights)
            
        return self
   
    def compute_p(self, X, S):
        q_low, q_high = self.get_q_kernel(X)
        p = ((q_low < S) & (S < q_high)).float()
        return p 
    
    def compute_loss(self, X, y, tau):
        G, log_det_L_stacked, q = self.get_frontiers(X, y)
        q_detached = q.detach()
        d = y.shape[-1]
        
        if self.loss_function == "CVAR": 
            p = (q < G).float()
            loss = ( p * G).mean()
        else:
            raise ValueError("The loss_function must be only_quantile or log_volume or full_volume or NLL.")
        
        with torch.no_grad():
            G_detached = G.detach()
            batch_coverage = (G_detached <= q_detached).float().mean() 
            
            scores = (G_detached / q_detached).squeeze()
            if scores.dim() == 0 or scores.size(0) < 2:
                valid_volume = torch.tensor(0.0, device=X.device)
            else:
                batch_q_alpha = torch.quantile(scores, tau)
                valid_q = q_detached * batch_q_alpha
                log_det_L_stacked, _ = self.get_det_L(X)

                log_vol_penalty = torch.logsumexp(-log_det_L_stacked, dim=1, keepdim=True)
                log_V_constant = math.log((math.pi ** (self.d / 2)) / math.gamma(self.d / 2 + 1))
                log_vol_k = log_V_constant + (self.d / 2) * torch.log(valid_q) + log_vol_penalty
                            
                valid_volume = torch.exp(log_vol_k).mean()
            
        return loss, batch_coverage, loss

    def compute_pb_loss(self, X, y, tau, with_coverage=False):
        G, _, q = self.get_frontiers(X, y)

        G_sg = G.detach()

        loss_q = (tau * F.relu(G_sg - q) + (1 - tau) * F.relu(q - G_sg)).mean()
        
        if with_coverage:
            coverage = (G_sg <= q).float().sum()
            return loss_q, coverage
        return loss_q
    
    def compute_pb_kernel_loss(self, X, y):
        q_low, q_high = self.get_q_kernel(X)
        S, _, _ = self.get_frontiers(X, y)
        G_sg = S.detach()

        tau_low, tau_high = self.tau_parameterAnnealer.step()

        loss_q_low = (tau_low * F.relu(G_sg - q_low) + (1 - tau_low) * F.relu(q_low - G_sg)).mean()
        loss_q_high = (tau_high * F.relu(G_sg - q_high) + (1 - tau_high) * F.relu(q_high - G_sg)).mean()
        
        return loss_q_low, loss_q_high



In [ ]:
class GeneratorD:
    def __init__(self, f, matrix_transform, dim_y, noise_type='gaussian', noise_std=1.0):
        self.f = f
        self.matrix_transform = matrix_transform
        self.noise_type = noise_type
        self.noise_std = noise_std
        self.dim_y = dim_y

    def _get_noise(self, n):
        if self.noise_type == 'gaussian':
            noise = torch.randn(n, self.dim_y)
        elif self.noise_type == 'uniform':
            noise = torch.rand(n, self.dim_y) * 2 - 1
        elif self.noise_type == 'exponential':
            # noise = torch.distributions.Exponential(rate=1.0).sample((n, self.dim_y))
            noise = torch.distributions.Exponential(rate=1.0).sample((n, self.dim_y)) - 1.0
        elif self.noise_type == 'multimodal':
            # noise = torch.distributions.Exponential(rate=1.0).sample((n, self.dim_y))
            noise1 = torch.distributions.Exponential(rate=1.0).sample((n, self.dim_y)) - 1.0
            noise2 = - torch.distributions.Exponential(rate=1.0).sample((n, self.dim_y)) - 3.0
            mask = (torch.rand(n, 1) > 0.5).float()
            noise = mask * noise1 + (1.0 - mask) * noise2
        else:
            raise ValueError(f"Type inconnu: {self.noise_type}")
        return (noise * self.noise_std).unsqueeze(2)

    def generate(self, n):
        x = 2 * torch.rand(n, 1) - 1
        fx = self.f(x)
        A_x = self.matrix_transform(x)
        noise = self._get_noise(n)
        correlated_noise = torch.bmm(A_x, noise).squeeze(2)
        y = fx + correlated_noise
        return x, y

    def generate_specific_y_given_x(self, x_tensor, n=1):
        x_repeated = x_tensor.repeat_interleave(n, dim=0)
        fx = self.f(x_repeated)
        A_x = self.matrix_transform(x_repeated)
        noise = self._get_noise(x_repeated.shape[0])
        correlated_noise = torch.bmm(A_x, noise).squeeze(2)
        y_flat = fx + correlated_noise
        return y_flat.view(n, self.dim_y)

def strange_matrix_transform_1D(x):
    n = x.shape[0]
    matrices = torch.eye(1).unsqueeze(0).repeat(n, 1, 1)
    matrices[:, 0, 0] = x.squeeze(-1)**2 + 0.5 
    return matrices

def circle_f_1D(x):
    return torch.sin(x*3)



class OracleModel:
    def __init__(self, generator, tau):
        self.gen = generator
        self.tau = tau

    def __call__(self, x):
        """
        Returns the exact ground-truth prediction that minimizes 
        the conditional tau-quantile of absolute error.
        """
        fx = self.gen.f(x)
        # Extract A(X) and ensure it has shape (batch_size, 1)
        Ax = self.gen.matrix_transform(x).view(x.shape[0], 1)
        
        # Oracle formula: f(X) - A(X) * (1 + 0.5 * ln(1 - tau))
        # Note: We multiply by noise_std to account for standard deviation scaling
        log_term = 1.0 + (math.log(1.0 - self.tau) / 2.0)
        
        return fx - (Ax * self.gen.noise_std * log_term)


def get_oracle_model(generator, tau):
    return OracleModel(generator, tau)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class QuantileMinimizationModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        # Simple feed-forward network for demonstration
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

def train_quantile_model_marginal(X_train, Y_train, tau=0.5, epochs=1000, batch_size=256, lr=1e-3):
    """
    Trains a model to minimize the empirical marginal tau quantile 
    of the absolute residuals |Y - f(X)| for each batch.
    """
    input_dim = X_train.shape[1]
    model = QuantileMinimizationModel(input_dim)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    model.train()
    num_samples = X_train.size(0)

    for epoch in range(epochs):
        # Shuffle data for stochastic mini-batch generation
        permutation = torch.randperm(num_samples)
        epoch_loss = 0.0
        batches = 0

        for i in range(0, num_samples, batch_size):
            indices = permutation[i:i+batch_size]
            batch_x, batch_y = X_train[indices], Y_train[indices]

            # 1. Forward pass
            predictions = model(batch_x)

            # 2. Compute absolute residuals: |Y - f(X)|
            # Ensure tensors are similarly shaped (batch_size, 1)
            abs_residuals = torch.abs(batch_y.view(-1, 1) - predictions.view(-1, 1))

            # 3. Compute the empirical marginal tau quantile of the batch
            # torch.quantile requires a 1D tensor for simple quantile extraction
            loss = torch.quantile(abs_residuals.squeeze(), q=tau)

            # 4. Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            batches += 1

        if epoch % 10 == 0:
            avg_loss = epoch_loss / batches
            print(f"Epoch {epoch:03d} | Avg Batch {tau}-Quantile of Abs Error: {avg_loss:.4f}")

    return model



In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

def plot_1d_comparison(
    X_train, 
    Y_train, 
    list_models, 
    list_name, 
    fontsize=20, 
    size_points=10, 
    name = None,
    color_points="#BAC1B8", 
    color_bound_points="#58A4B0", 
    line_colors=None,
    line_styles=None
):
    """
    Plots the 1D training data and the models' central predictions f(X).
    Incorporates varying line styles for improved color-blind accessibility.
    """
    # Define fallback colors matching the warm palette of the target script
    if line_colors is None:
        line_colors = ["#D62828", "#F77F00", "#1D3557", "#457B9D"]
        
    # Define fallback line styles for accessibility (solid, dashed, dash-dot, dotted)
    if line_styles is None:
        line_styles = ["-", "--", "-.", ":"]
        
    # 1. Create a dense grid of X values for smooth plotting
    x_min, x_max = X_train.min().item(), X_train.max().item()
    X_grid = torch.linspace(x_min, x_max, 500).view(-1, 1)

    fig, ax = plt.subplots(figsize=(10, 6))

    # Convert tensors to numpy arrays for matplotlib
    X_train_np = X_train.squeeze().numpy()
    Y_train_np = Y_train.squeeze().numpy()
    
    # Plot the raw training data
    ax.scatter(X_train_np, Y_train_np, 
                color=color_points, alpha=0.5, s=size_points, 
                edgecolors=color_bound_points, linewidths=0.5, zorder=1)
                
    # Initialize legend handles
    handles = []
    scatter_handle = plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color_points, 
                                markeredgecolor=color_bound_points, markersize=8, alpha=0.75, label='Test points')
    handles.append(scatter_handle)

    # 2. Generate predictions
    X_grid_np = X_grid.squeeze().numpy()
    
    for idx, model in enumerate(list_models):
        with torch.no_grad():
            Y_grid_pred = model(X_grid)
            Y_grid_pred_np = Y_grid_pred.squeeze().numpy()
            
            # Select line color and style cyclically to prevent out-of-bounds errors
            c = line_colors[idx % len(line_colors)]
            s = line_styles[idx % len(line_styles)]

            # Plot the central prediction line f(X)
            ax.plot(X_grid_np, Y_grid_pred_np, 
                    linewidth=2.5, color=c, linestyle=s, zorder=2)
            
            # Append handles for this model's line
            line_handle = plt.Line2D([0], [0], color=c, linestyle=s, linewidth=2.5, label=f"{list_name[idx]}")
            handles.append(line_handle)

    # --------------------------------------------------
    # 3. Formatting and Axes
    # --------------------------------------------------
    ax.set_xlabel("$x$", fontsize=fontsize)
    ax.set_ylabel("$y$", fontsize=fontsize)
    
    # Tick size formatting
    ax.tick_params(axis='both', which='major', labelsize=fontsize - 2)

    # Grid formatting
    ax.grid(True, linestyle='--', alpha=0.4, zorder=0)

    # Legend formatting
    ax.legend(
        handles=handles,
        fontsize=fontsize - 2, 
        loc='best', 
        framealpha=0.9, 
        edgecolor='lightgrey'
    )
    
    # Apply Limits
    ax.set_xlim(x_min, x_max)
    
    plt.tight_layout()

    if name is not None:
        plt.savefig(f"../figs/median_{name}.pdf", dpi=300)
    plt.show()

class WrapperModel:
    def __init__(self, model):
        self.model = model

    def __call__(self, X):
        mu_vals = self.model(X)[0]
        return mu_vals

In [ ]:

# ==========================================
# 3. Entraînement
# ==========================================
torch.manual_seed(42)

generator = GeneratorD(
    f=circle_f_1D,
    matrix_transform=strange_matrix_transform_1D,
    dim_y=1,
    noise_std=1.0,
    noise_type="exponential"
)

# On génère un dataset d'entraînement
N_train = 3000
X_train, Y_train = generator.generate(N_train)
X_val, Y_val = generator.generate(N_train)
X_calibration, Y_calibration = generator.generate(N_train)
X_test, Y_test = generator.generate(N_train)

dim_X = 1
dim_y = 1

tau = 0.50

In [ ]:


model_marginal = train_quantile_model_marginal(X_train, Y_train, tau=tau, batch_size=512)

In [ ]:
   

batch_size = 256
num_epochs = 1_000
lr = 5e-4


tau_param = TauParameterAnnealer(tau,                
                 warm_start_step=500, 
                 tau_low_target_step=100, 
                 tau_low_steepness=1e-3,
                 tau_high_target_step=100, 
                 tau_high_steepness=1e-2,
                 low_error_init=0.5,   
                 low_error_max=0.01,    
                 high_error_init=0.2,  
                 high_error_max=0.01,  
                 eps=1e-5              
                 )


model_ours = UnifiedConditionalEstimator(dim_X=dim_X, dim_y=dim_y, 
                                    cov_mode="full_cholesky", num_flow_layers=0, K=1,
                                    det_normalized=True
                                    )


model_ours.fit(X_train, Y_train, 
          tau=tau, 
          epochs=num_epochs, 
          lr=lr, 
          batch_size=batch_size, 
          return_best=True, 
          print_every=10,
          tau_parameterAnnealer=tau_param,
          loss_function="log_volume"
          )


In [ ]:
   

batch_size = 256
num_epochs = 1_000
lr = 5e-4


tau_param = TauParameterAnnealer(tau,                
                 warm_start_step=500, 
                 tau_low_target_step=100, 
                 tau_low_steepness=1e-3,
                 tau_high_target_step=100, 
                 tau_high_steepness=1e-2,
                 low_error_init=0.5,   
                 low_error_max=0.01,    
                 high_error_init=0.2,  
                 high_error_max=0.01,  
                 eps=1e-5              
                 )


model_CVAR = UnifiedConditionalEstimatorCVAR(dim_X=dim_X, dim_y=dim_y, 
                                    cov_mode="full_cholesky", num_flow_layers=0, K=1,
                                    det_normalized=True
                                    )


model_CVAR.fit(X_train, Y_train, 
          tau=tau, 
          epochs=num_epochs, 
          lr=lr, 
          batch_size=batch_size, 
          return_best=True, 
          print_every=10,
          tau_parameterAnnealer=tau_param,
          loss_function="CVAR"
          )



In [ ]:

    
wrapper_ours = WrapperModel(model_ours)
wrapper_CVAR = WrapperModel(model_CVAR)

In [ ]:



# 1. Instantiate the Oracle
oracle = get_oracle_model(generator, tau=tau)


In [ ]:
import matplotlib
import os
os.environ["PATH"] += os.pathsep + "/Library/TeX/texbin"

matplotlib.use('agg')
# matplotlib.use('pdf')
matplotlib.rcParams.update({
    "pgf.texsystem": "pdflatex",
    'font.family': 'serif',
    'font.size': 20,
    'text.usetex': True,
    'pgf.rcfonts': False,
    # 'legend.framealpha': 0.5,
    'text.latex.preamble': r'\usepackage{times} \usepackage{amsmath} \usepackage{amsfonts} \usepackage{amssymb} \usepackage{xcolor}'
})


In [ ]:
plot_1d_comparison(X_train, 
                    Y_train, 
                    [oracle, wrapper_ours, wrapper_CVAR, model_marginal],  
                    ["Oracle" ,"SLS", "CVaR", "Marginal"],  
                    fontsize=20, 
                    size_points=10,
                    color_points="#BAC1B8",
                    color_bound_points="#58A4B0",
                    name="left"
                    )

In [ ]:
def strange_matrix_transform_1D(x):
    n = x.shape[0]
    matrices = torch.eye(1).unsqueeze(0).repeat(n, 1, 1)
    matrices[:, 0, 0] = x.squeeze(-1)**2 + 0.5 
    return matrices

def circle_f_transformed(x):
    return torch.sin(x*3) + 4*(x > 0.5)



# ==========================================
# 3. Entraînement
# ==========================================
torch.manual_seed(42)

generator = GeneratorD(
    f=circle_f_transformed,
    matrix_transform=strange_matrix_transform_1D,
    dim_y=1,
    noise_std=1.0,
    noise_type="exponential"
)

# On génère un dataset d'entraînement
N_train = 3000
X_train, Y_train = generator.generate(N_train)
X_val, Y_val = generator.generate(N_train)
X_calibration, Y_calibration = generator.generate(N_train)
X_test, Y_test = generator.generate(N_train)

dim_X = 1
dim_y = 1

tau = 0.50

In [ ]:
model_marginal = train_quantile_model_marginal(X_train, Y_train, tau=tau, batch_size=512)

   

batch_size = 256
num_epochs = 1_000
lr = 5e-4


tau_param = TauParameterAnnealer(tau,                
                 warm_start_step=500, 
                 tau_low_target_step=100, 
                 tau_low_steepness=1e-3,
                 tau_high_target_step=100, 
                 tau_high_steepness=1e-2,
                 low_error_init=0.5,   
                 low_error_max=0.01,    
                 high_error_init=0.2,  
                 high_error_max=0.01,  
                 eps=1e-5              
                 )


model_ours = UnifiedConditionalEstimator(dim_X=dim_X, dim_y=dim_y, 
                                    cov_mode="full_cholesky", num_flow_layers=0, K=1,
                                    det_normalized=True
                                    )


model_ours.fit(X_train, Y_train, 
          tau=tau, 
          epochs=num_epochs, 
          lr=lr, 
          batch_size=batch_size, 
          return_best=True, 
          print_every=10,
          tau_parameterAnnealer=tau_param,
          loss_function="log_volume"
          )


   

batch_size = 256
num_epochs = 1_000
lr = 5e-4


tau_param = TauParameterAnnealer(tau,                
                 warm_start_step=500, 
                 tau_low_target_step=100, 
                 tau_low_steepness=1e-3,
                 tau_high_target_step=100, 
                 tau_high_steepness=1e-2,
                 low_error_init=0.5,   
                 low_error_max=0.01,    
                 high_error_init=0.2,  
                 high_error_max=0.01,  
                 eps=1e-5              
                 )


model_CVAR = UnifiedConditionalEstimatorCVAR(dim_X=dim_X, dim_y=dim_y, 
                                    cov_mode="full_cholesky", num_flow_layers=0, K=1,
                                    det_normalized=True
                                    )


model_CVAR.fit(X_train, Y_train, 
          tau=tau, 
          epochs=num_epochs, 
          lr=lr, 
          batch_size=batch_size, 
          return_best=True, 
          print_every=10,
          tau_parameterAnnealer=tau_param,
          loss_function="CVAR"
          )

model_CVAR.conformalize(X_calibration, Y_calibration, tau, fake_for_trial=True)
vol = model_CVAR.compute_average_volume(X_test)
print(f"Volume de la région de confiance: {vol.item():.4f}")

plot_1d_conditional_contours(generator, model_CVAR, f=circle_f_1D)

oracle = get_oracle_model(generator, tau=tau)

wrapper_ours = WrapperModel(model_ours)
wrapper_CVAR = WrapperModel(model_CVAR)

In [ ]:
wrapper_ours = WrapperModel(model_ours)
wrapper_CVAR = WrapperModel(model_CVAR)

plot_1d_comparison(X_train, 
                    Y_train, 
                    [oracle, wrapper_ours, wrapper_CVAR, model_marginal],  
                    ["Oracle" ,"SLS", "CVaR", "Marginal"],  
                    fontsize=20, 
                    size_points=10,
                    color_points="#BAC1B8",
                    color_bound_points="#58A4B0",
                    name="right"
                    )